# BIO và mẫu toàn thông báo trong train

Truy hồi mẫu toàn văn để chuyển cụm và liên kết. Cặp chưa chuyển được dùng mẫu câu, BIO và bộ nối ngữ cảnh.

[Báo cáo tổng hợp](../reports/tong-hop-pipeline-smartnotice-v3.3-2026-11-23.md) · [Giao thức đánh giá](../../../docs/evaluation.md)

## 1. Cấu hình và dữ liệu

Nạp thông báo đã chuẩn bị và kiểm tra hash, nhóm, schema và các khoảng ký tự. Tham chiếu test v3.3 là chú giải lịch sử có máy hỗ trợ.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tools/compare_notice_pipelines.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Mở notebook từ workspace SmartNotice.")
os.chdir(repo)
split = Path(os.environ.get("SMARTNOTICE_SPLIT_ROOT", "tmp/drive-run-20261123-fresh")).resolve()
base = Path(os.environ.get("SMARTNOTICE_BAMIBERT_DIR", "tmp/bamibert-checkpoint")).resolve()
destination = Path(os.environ.get(
    "SMARTNOTICE_NOTEBOOK_OUTPUT", "reports/generated/pipeline-notebooks-v33-20261123"
)).resolve()
baseline = Path(os.environ.get(
    "SMARTNOTICE_BASELINE", "reports/generated/bamibert/v3_3-2026-11-23-final-seed42"
)).resolve()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def run_tool(script, *arguments, **environment):
    env = {**os.environ, "PYTHONHASHSEED": "42", "TOKENIZERS_PARALLELISM": "false", **environment}
    result = subprocess.run(
        [sys.executable, "-u", str(repo / "tools" / script), *map(str, arguments)],
        cwd=repo, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode:
        print(result.stdout[-4000:], result.stderr[-4000:])
        result.check_returncode()
    print(result.stdout.strip())

In [2]:
from smartnotice.modeling.data import labeled_links, load_development, sha256, span_key

development, provenance = load_development(split)
for part, docs in development.items():
    print(part, {
        "notices": len(docs),
        "groups": len({d.group_id for d in docs}),
        "typed_spans": sum(len({span_key(s) for s in d.annotation.spans}) for d in docs),
        "links": sum(len(set(labeled_links(d.annotation))) for d in docs),
        "events": sum(len(d.annotation.events) for d in docs),
    })
print("Phân bổ:", provenance["split_counts"], "gold_ready:", provenance["gold_ready"])

train {'notices': 403, 'groups': 281, 'typed_spans': 4664, 'links': 3847, 'events': 1516}
validation {'notices': 93, 'groups': 77, 'typed_spans': 1078, 'links': 873, 'events': 357}
Phân bổ: {'train': 403, 'test': 126, 'validation': 93, 'domain_test': 62} gold_ready: False


## 2. Huấn luyện từ train và chọn ngưỡng validation

Các lớp dựng mọi chỉ mục từ train. Ngưỡng liên kết tối ưu F1 sự kiện, rồi F1 liên kết khi hòa.

In [3]:
from smartnotice.modeling.experiment import write_summary
from smartnotice.modeling.metrics import prf


def show(scores):
    span = scores["span"]["micro"]
    link = scores["link"]["micro"]
    event = scores["event"]
    for caption, score in (("Cụm có kiểu", span), ("Quan hệ", link), ("Sự kiện", event)):
        print(
            f"{caption}: F1 {score['f1']:.2%} | P {score['precision']:.2%} | "
            f"R {score['recall']:.2%} | đúng/dự đoán/tham chiếu "
            f"{score['matched']}/{score['predicted']}/{score['expected']}"
        )
    print("Ngày tường minh:", scores["dates"]["explicit"])


def without_reference(docs):
    from smartnotice.modeling.data import Document
    from smartnotice.schemas import Annotation

    return [
        Document(d.notice, Annotation(
            doc_id=d.notice.doc_id, annotator_id="inference", spans=[], events=[]
        ), d.group_id)
        for d in docs
    ]


def boundary_diagnostics(docs, predictions):
    expected, predicted = {}, {}
    for d in docs:
        doc_id = d.notice.doc_id
        for target, spans in (
            (expected, map(span_key, d.annotation.spans)), (predicted, predictions[doc_id][0])
        ):
            for begin, end, kind in spans:
                target.setdefault((doc_id, begin, end), set()).add(kind)
    matched = expected.keys() & predicted.keys()
    return {
        "untyped": prf(set(expected), set(predicted)),
        "correct_type_sets": sum(expected[key] == predicted[key] for key in matched),
        "matched_boundaries": len(matched),
    }

In [4]:
from smartnotice.modeling.contextual import ContextExtractor, TemplateExtractor, DocumentTemplateExtractor
from tools.compare_notice_pipelines import _context_predictions, _metrics
from smartnotice.preprocessing.split import DEFAULT_SEED

seed = DEFAULT_SEED

out = destination / "run-06"
out.mkdir(parents=True, exist_ok=False)
tagger = ContextExtractor(seed)
tagger.fit(development["train"])
sentences = TemplateExtractor(tagger, similarity=0.65)
sentences.fit(development["train"])
model = DocumentTemplateExtractor(sentences, similarity=0.60)
model.fit(development["train"])

In [5]:
import joblib

valid = development["validation"]
grid = {
    cutoff: _metrics(valid, pred)
    for cutoff, pred in _context_predictions(model, without_reference(valid), (0.35, 0.50, 0.65)).items()
}
selected = max(grid, key=lambda c: (grid[c]["event"]["f1"], grid[c]["link"]["micro"]["f1"]))
print("Ngưỡng liên kết đã chốt:", selected)
show(grid[selected])
joblib.dump(model, out / "model.joblib", compress=3)
write_summary(out / "selection.json", {
    "seed": seed, "data_hashes": provenance["hashes"], "link_cutoff": selected, "validation_grid": grid,
})

Ngưỡng liên kết đã chốt: 0.65
Cụm có kiểu: F1 18.44% | P 19.81% | R 17.25% | đúng/dự đoán/tham chiếu 186/939/1078
Quan hệ: F1 10.16% | P 16.54% | R 7.33% | đúng/dự đoán/tham chiếu 64/387/873
Sự kiện: F1 3.62% | P 3.92% | R 3.36% | đúng/dự đoán/tham chiếu 12/306/357
Ngày tường minh: {'expected_links': 135, 'matched_links': 14, 'correct_dates': 12, 'matched_date_accuracy': 0.8571428571428571, 'matched_inferred_flag_accuracy': 1.0, 'end_to_end_date_accuracy': 0.08888888888888889}


## 3. Suy luận với văn bản test và chấm điểm

In [6]:
from tools.compare_notice_pipelines import _metrics, _test_docs

test = _test_docs(split)
print("Test:", len(test), "thông báo,", len({d.group_id for d in test}), "nhóm")

Test: 126 thông báo, 118 nhóm


In [7]:
predictions = _context_predictions(model, without_reference(test), (selected,))[selected]
result = _metrics(test, predictions)
result["boundaries"] = boundary_diagnostics(test, predictions)
show(result)
print("Ranh giới và gán kiểu có điều kiện:", result["boundaries"])
write_summary(out / "test_scores.json", result)

Cụm có kiểu: F1 19.54% | P 21.10% | R 18.20% | đúng/dự đoán/tham chiếu 257/1218/1412
Quan hệ: F1 9.97% | P 15.67% | R 7.31% | đúng/dự đoán/tham chiếu 84/536/1149
Sự kiện: F1 1.48% | P 1.75% | R 1.28% | đúng/dự đoán/tham chiếu 6/343/468
Ngày tường minh: {'expected_links': 188, 'matched_links': 20, 'correct_dates': 16, 'matched_date_accuracy': 0.8, 'matched_inferred_flag_accuracy': 1.0, 'end_to_end_date_accuracy': 0.0851063829787234}
Ranh giới và gán kiểu có điều kiện: {'untyped': {'precision': 0.21187139323990106, 'recall': 0.1820113314447592, 'f1': 0.19580952380952382, 'matched': 257, 'predicted': 1213, 'expected': 1412}, 'correct_type_sets': 253, 'matched_boundaries': 257}


Bộ phân loại áp dụng dùng trọng số đã huấn luyện ở notebook 01 và cùng panel hồ sơ tổng hợp.

In [8]:
if os.environ.get("SMARTNOTICE_SKIP_APPLICABILITY") == "1":
    print("Applicability evaluation disabled for this run.")
else:
    import torch
    from transformers import AutoTokenizer

    torch.set_num_threads(2)
    device = torch.device(os.environ.get("SMARTNOTICE_APP_DEVICE", "cpu"))
    tokenizer = AutoTokenizer.from_pretrained(base, use_fast=True)
    app_run = baseline

    record = read_json(app_run / "run.json")
    assert record["data"]["hashes"] == provenance["hashes"]
    assert record["model_weights_sha256"] == sha256(base / "model.safetensors")
    assert record["tokenizer_sha256"] == sha256(base / "tokenizer.json")

    from types import SimpleNamespace
    from tools.compare_notice_pipelines import _app

    app = _app(
        SimpleNamespace(base=base, baseline=app_run),
        development["train"], test, {"pipeline": predictions}, device, tokenizer,
    )["pipeline"]
    result["applicability"] = app
    print("F1 sự kiện và nhãn áp dụng:", f"{app['joint_exact_event_profile']['f1']:.2%}")
    write_summary(out / "test_scores.json", result)

F1 sự kiện và nhãn áp dụng: 0.87%
